# Sort and cluster TimePix events

Raw HDF5 files → one sequential read → toa-sorted *parts* in memory → clusters (≈ one row per electron), in a single pass.
Peak memory (~1.4 GB for 32 files) is set by the part size, not by the dataset size.

Vocabulary: an **event** is one pixel record (x, y, toa, tot); a **part** is a dict of event columns sorted by toa;
a **cluster** is a group of events from one electron. See the README for the layout of the package.

In [ ]:
import time
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from kronos import (cluster_parts, find_files, load_parts, merge_sort,
                    neighbor_time_gaps, save_parts, time_disorder)

%matplotlib inline

## 1. Choose the files

In [ ]:
experiment = "austin_test"            # e.g. "TestSeq1_1k" on the external SSD
data_directory = Path("/Users/7c2/Desktop/Projects/TimePix/data/2026_09_30_tp4_firsttry")
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
work_directory = project_root / "outputs" / experiment

KEEP_SORTED_EVENTS = False            # True also saves the sorted events (slower)

files = find_files(data_directory, f"{experiment}_*.h5")
print(len(files), "files")
files[:3]

## 2. Look at the raw files
Only small slices are read. `disorder_ticks` is the largest step back in toa within the first
million rows: well below `max_disorder` (65536 ticks) means `merge_sort` works; comparable to the
file's whole span means the files are shuffled and `bucket_sort` is needed.
`time_slots` shows which round-robin 16384-tick time blocks each file holds.

In [ ]:
with h5py.File(files[0], "r") as h5:
    toa = h5["toa"]
    print(f"rows={len(toa):,}  dtype={toa.dtype}  h5 chunks={toa.chunks}  "
          f"compression={toa.compression}")

rows = []
for path in files:
    with h5py.File(path, "r") as h5:
        head = h5["toa"][:1_000_000]
        rows.append({"file": path.name, "events": len(h5["toa"]),
                     "toa_first": int(head[0]), "toa_last": int(h5["toa"][-1]),
                     "disorder_ticks": time_disorder(head),
                     "time_slots": np.unique((head >> 14) % 16).tolist(),
                     "x_max": int(h5["x"][:1_000_000].max()),
                     "y_max": int(h5["y"][:1_000_000].max())})
layout = pd.DataFrame(rows)
layout

## 3. Choose the cluster parameters
`merge_sort` yields sorted parts lazily, so `next()` sorts just the first one.
For every event: the time since the latest earlier event on a neighboring pixel.
Events from one electron form a narrow peak at small gaps; chance coincidences give a flat floor.
Set `DT_MAX` where the peak meets the floor.

In [ ]:
sample = next(merge_sort(files))
gaps = neighbor_time_gaps(sample["x"], sample["y"], sample["toa"])

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(gaps[(gaps >= 0) & (gaps < 100)], bins=100, range=(0, 100), log=True)
ax.set(xlabel="toa ticks to previous neighboring event", ylabel="events")
plt.show()

In [ ]:
DT_MAX = 2    # toa ticks; austin_test: peak at 0-1 ticks, flat floor from 2 on
RADIUS = 1    # pixels (square neighborhood)

## 4. Sort and cluster in one pass
If `merge_sort` raises `ValueError`, the files are more jumbled than `max_disorder`:
use `bucket_sort(files, work_directory / "tmp")` in its place.

In [ ]:
cluster_directory = work_directory / "clusters"   # delete it to run again
sorted_directory = work_directory / "sorted"

if cluster_directory.exists():
    print("already clustered")
else:
    start = time.perf_counter()
    parts = merge_sort(files)
    if KEEP_SORTED_EVENTS:
        save_parts(parts, sorted_directory)
        parts = load_parts(sorted_directory)
    cluster_parts(parts, cluster_directory, DT_MAX, RADIUS)
    seconds = time.perf_counter() - start
    print(f"{seconds:.0f} s, {layout.events.sum() / seconds / 1e6:.1f} M events/s")

In [ ]:
# Check: every event ended up in exactly one cluster.
tables = sorted(cluster_directory.glob("clusters-*.parquet"))
n_events = sum(pd.read_parquet(t, columns=["n_events"]).n_events.sum() for t in tables)
assert n_events == layout.events.sum()
print(f"✓ {n_events:,} events in {len(tables)} cluster tables")

## 5. First look at the clusters
There are ~0.5 clusters per event, so for large runs the full table does not fit in memory:
look at the first few tables.

In [ ]:
clusters = pd.concat([pd.read_parquet(t) for t in tables[:2]], ignore_index=True)
print(f"{len(clusters):,} clusters from {clusters.n_events.sum():,} events")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(clusters.n_events, bins=np.arange(1, 12), log=True)
axes[0].set(xlabel="events per cluster", ylabel="clusters")
axes[1].hist(clusters.tot, bins=100, range=(0, 100), log=True)   # 2047 = saturated
axes[1].set(xlabel="summed ToT per cluster")
image, _, _ = np.histogram2d(clusters.y, clusters.x, bins=(512, 448),
                             range=((0, 512), (0, 448)))
axes[2].imshow(image, origin="lower", cmap="magma",
               vmax=np.percentile(image, 99.9))   # clip hot pixels
axes[2].set(title="cluster centroids")
plt.show()